# Analytics 1 - Firme din Romania (L2)

Notebook de analiza pe datele consolidate L2 (`data.gov.ro/l2_data/`), produse in `data-download-firme-rom.ipynb` din:
- `OD_FIRME` + `OD_CAEN_AUTORIZAT` + `OD_STARE_FIRMA` (Registrul Comertului)
- situatiile financiare `ir` / `uu` / `bl_bs_sl` (data.gov.ro, Guvernul Romaniei)

Fisierele disponibile in `l2_data/`:
- `ir_l2.csv`, `uu_l2.csv`, `bl_bs_sl_l2.csv` - date financiare + identitatea firmei, un rand per CUI
- `caen_autorizat_l2.csv` - coduri CAEN autorizate per firma (neagregat), legat prin `COD_INMATRICULARE`
- `stare_firma_l2.csv` - coduri de stare per firma (neagregat), legat prin `COD_INMATRICULARE`

In [ ]:
import IPython
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1100)
pd.set_option('display.width', 1000)
# Disable the SettingWithCopyWarning
pd.options.mode.chained_assignment = None

from itables import init_notebook_mode
import itables.options

itables.options.lengthMenu = [5, 10, 15, 25, 50]
init_notebook_mode(all_interactive=True)

import numpy as np
import matplotlib.pyplot as plt

from IPython.display import HTML, display

# Tema inchisa pentru tabelele itables: fundal negru, text alb, headere/sageti de sortare galbene.
# Sortarea pe coloane ramane activa implicit (DataTables) - click pe header pentru asc/desc.
# Acopera atat clasele DataTables 1.x (dataTables_*) cat si cele 2.x (dt-*), pt compatibilitate.
display(HTML("""
<style>
table.dataTable, table.dataTable thead, table.dataTable tbody,
table.dataTable thead th, table.dataTable thead td,
table.dataTable tbody th, table.dataTable tbody td {
    background-color: #000000 !important;
    color: #ffffff !important;
    border-color: #444444 !important;
}

table.dataTable thead th, table.dataTable thead td {
    color: #ffd700 !important;
}

table.dataTable tbody tr:hover td {
    background-color: #222222 !important;
}

.dataTables_wrapper, .dt-container,
.dataTables_wrapper .dataTables_info, .dt-container .dt-info,
.dataTables_wrapper .dataTables_length, .dt-container .dt-length,
.dataTables_wrapper .dataTables_filter, .dt-container .dt-search,
.dataTables_wrapper .dataTables_paginate, .dt-container .dt-paging {
    color: #ffffff !important;
}

.dataTables_wrapper .dataTables_filter input, .dt-container .dt-search input,
.dataTables_wrapper .dataTables_length select, .dt-container .dt-length select,
.dt-input {
    background-color: #000000 !important;
    color: #ffffff !important;
    border: 1px solid #666666 !important;
}

.dataTables_wrapper .dataTables_paginate .paginate_button,
.dt-container .dt-paging .dt-paging-button {
    color: #ffffff !important;
    background: transparent !important;
    border-color: #444444 !important;
}

.dataTables_wrapper .dataTables_paginate .paginate_button.current,
.dt-container .dt-paging .dt-paging-button.current {
    color: #000000 !important;
    background: #ffd700 !important;
    border-color: #ffd700 !important;
}

.dataTables_wrapper .dataTables_paginate .paginate_button.disabled,
.dt-container .dt-paging .dt-paging-button.disabled {
    color: #666666 !important;
}
</style>
"""))

## Incarcare date L2

Celula de mai jos doar **pregateste** caile si un loader generic - nu citeste niciun fisier inca. Incarca punctual, cand ai nevoie, cu `incarca_l2("cheie")` (sau `incarca_l2("cheie", nrows=...)` pentru un esantion rapid din fisierele mari).

In [ ]:
from pathlib import Path

L2_DIR = Path("/Users/tudor/Documents/Data-for-Projects/Cercetare-Research/data.gov.ro/l2_data")

# cheie -> nume fisier in L2_DIR
L2_FILES = {
    "ir": "ir_l2.csv",
    "uu": "uu_l2.csv",
    "bl_bs_sl": "bl_bs_sl_l2.csv",
    "caen_autorizat": "caen_autorizat_l2.csv",
    "stare_firma": "stare_firma_l2.csv",
}

# Cache in memorie, ca sa nu recitim de pe disc daca apelam incarca_l2 de mai multe ori pentru aceeasi cheie.
l2_data = {}


def incarca_l2(cheie: str, forte_recitire: bool = False, **read_csv_kwargs) -> pd.DataFrame:
    """Incarca (si cacheaza) un fisier din L2_DIR intr-un DataFrame. Nu face nimic pana nu e apelata explicit."""
    if cheie not in L2_FILES:
        raise KeyError(f"Cheie necunoscuta '{cheie}'. Chei disponibile: {list(L2_FILES)}")
    if forte_recitire or cheie not in l2_data:
        path = L2_DIR / L2_FILES[cheie]
        l2_data[cheie] = pd.read_csv(path, encoding="utf-8-sig", **read_csv_kwargs)
    return l2_data[cheie]


print("Fisiere L2 disponibile:", list(L2_FILES))
print("Niciun fisier incarcat inca - foloseste incarca_l2(cheie) cand ai nevoie de un anumit tabel.")